Run this notebook on CPU from the repository root or `analysis/`.

By default it plots the paper's descriptor similarity matrix from `analysis/paper_data/context_similarity_matrix.json`. To plot your own extraction, run `pixi run python -m analysis.context_similarity --output logs/similarity` and set `matrix_file` below to the `result.json` in that directory. Exports use a fresh `logs/paper_figures/` directory.

In [ ]:
%matplotlib inline
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.transforms import offset_copy

root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from analysis.artifacts import create_output

plt.rcdefaults()
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 20, 'axes.linewidth': 1,
                     'xtick.major.size': 3, 'ytick.major.size': 3,
                     'xtick.major.width': 1, 'ytick.major.width': 1,
                     'xtick.major.pad': 3.5, 'ytick.major.pad': 3.5, 'pdf.fonttype': 3})

config_file = Path('conf/analysis/context_similarity.json')
matrix_file = Path('analysis/paper_data/context_similarity_matrix.json')


## Read the similarity matrix

Read the 8-by-8 matrix and the CoM coordinates on CPU.

In [ ]:
config = json.loads(config_file.read_text())
matrix = np.asarray(json.loads(matrix_file.read_text())['matrix'])
coordinates = np.asarray(config['coordinates'])
if matrix.shape != (8, 8) or coordinates.shape != (8, 2):
    raise ValueError('Expected an 8-by-8 similarity matrix and eight XY coordinates')
if not np.isfinite(matrix).all() or not np.isfinite(coordinates).all():
    raise ValueError('Similarity and coordinates must be finite')
inputs = {'matrix': str(matrix_file.resolve()), 'config': str(config_file.resolve())}
print('Similarity:', matrix.shape, '| Coordinates:', coordinates.shape)

run_dir = create_output(root / 'logs/paper_figures' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'), {'inputs': inputs})

## Plot and export

Keep the paper layout and export PDF/PNG and plotted values into a fresh output directory.


In [ ]:
page_width, page_height = 773.96625, 358.56
panel_size = 329.820625
fig = plt.figure(figsize=(page_width / 72, page_height / 72), dpi=100)
location = fig.add_axes([56.5025 / page_width, 27.299375 / page_height,
                         panel_size / page_width, panel_size / page_height])
heatmap = fig.add_axes([442.705625 / page_width, 27.299375 / page_height,
                        panel_size / page_width, panel_size / page_height])
outline = [(-1, -1), (1, -1), (1, -.5), (.25, -.5),
           (.25, 1), (-.25, 1), (-.25, -.5), (-1, -.5)]
location.add_patch(Polygon(outline, facecolor='#8ea3b7', edgecolor='#778899', linewidth=1.6))
location.set(xlim=(-1.18, 1.18), ylim=(1.18, -1.18), aspect='equal')
location.set_xticks(np.arange(-1, 1.1, .5))
location.set_yticks(np.arange(-1, 1.1, .5))
labels = [f'P{i + 1}' for i in range(8)]
colors = [plt.get_cmap('tab10')(i // 2) for i in range(8)]
for label, xy, color in zip(labels, coordinates, colors):
    location.scatter(*xy, s=210, color=color, edgecolors='black', linewidths=1.2, zorder=3)
    location.annotate(label, xy, xytext=(7, 5), textcoords='offset points',
                      fontsize=20, fontweight='bold', zorder=4)
heatmap.imshow(matrix, cmap='RdBu_r', vmin=-1, vmax=1, interpolation='nearest')
heatmap.set_xticks(range(8), labels)
heatmap.set_yticks(range(8), labels)
for ticks in (heatmap.get_xticklabels(), heatmap.get_yticklabels()):
    for tick, color in zip(ticks, colors):
        tick.set_color(color)
        tick.set_fontweight('bold')
# Preserve the published Matplotlib 3.5 text baseline.
text_transform = offset_copy(heatmap.transData, fig=fig, y=-.1328125, units='points')
for (i, j), value in np.ndenumerate(matrix):
    heatmap.text(j, i, f'{value:.2f}', ha='center', va='center',
                 color='white' if abs(round(value, 2)) > .6 else 'black', fontsize=12.5,
                 fontweight='bold', transform=text_transform)
fig.savefig(run_dir / '8-ctx-token.pdf', dpi=100)
fig.savefig(run_dir / '8-ctx-token.png', dpi=200)
np.savetxt(run_dir / 'similarity.csv', matrix, delimiter=',')
plt.show()
print(run_dir)


Paper styling is retained; plotted values come from the selected inputs. PDF file bytes may differ because of metadata.
